In [2]:
# read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [5]:
len(text)

1115394

In [10]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [11]:
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [13]:
# let's now encode the entire text dataset and store it into a torch.Tensor
import torch # we use PyTorch: https://pytorch.org
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000]) # the 1000 characters we looked at earier will to the GPT look like this

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [14]:
# Let's now split up the data into train and validation sets
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [15]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [16]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(context, target)

tensor([18]) tensor(47)
tensor([18, 47]) tensor(56)
tensor([18, 47, 56]) tensor(57)
tensor([18, 47, 56, 57]) tensor(58)
tensor([18, 47, 56, 57, 58]) tensor(1)
tensor([18, 47, 56, 57, 58,  1]) tensor(15)
tensor([18, 47, 56, 57, 58,  1, 15]) tensor(47)
tensor([18, 47, 56, 57, 58,  1, 15, 47]) tensor(58)


In [20]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44, 53

In [30]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets = None):
        logits = self.token_embedding_table(idx) # B, T, C
        if targets is None:
            loss = None
        else:
            loss = F.cross_entropy(logits.view(-1, vocab_size), targets.view(-1))
        return logits, loss

    def generate(self, idx, max_new_tokens):
        # max_new_tokens -> number of token that we will generate
        # idx is input tkens. based on this, what do you generate next?
        # as this is just a bigram model - for each batch, I just need the last input right? so my output will be of share B, max_new_tokens
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :] # B, C
            #get probabilites from softmax
            probs = F.softmax(logits, dim = 1) # (B, C)
            idx_next = torch.multinomial(probs, num_samples=1) #(B, 1)
            idx = torch.cat((idx, idx_next), dim = 1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)

print(loss)

generated_batches = m.generate(idx = torch.zeros((xb.shape[0],1), dtype = torch.long), max_new_tokens = 100)

for generated_batch in generated_batches:
    print(decode(generated_batch.tolist()))



tensor(4.8786, grad_fn=<NllLossBackward0>)

Sqcot?p.k&lFhF$bjuDnmW-jKppY,3&YxfFJZgXXQq-LKuC z3SqhzkhJrQ!PmU?WWnPgZcbVTbdtt$Rlv$ktORIs&duXY,SU'Pl

STET:CERjqPyjKuLehVnlgFEj?aZR
JW: f$etNXrFCkRr:keviHkdbfXiyJ?GrnmaSqbWhsug!uxhOLasi
pNJApJq-AUA'zeha

S.LP q3SK;wwf?EwXya!weDOj:&oibo-zoT;lxzUYIBTiq.DqVlv&vv'TbHxim'zoIM?a!vnE
o fTXiq-Ya!Pgc;gm,evj?q-wO

JWMvvn3!.jgCMj3Sx;
SVjusJBNNOpM,ARppxl,i.i-Yg.qfN:BiPcnZALPsqHEaoiNVRyF-oALETlxj3SV?3:rmZY:3r rDCIo!


In [35]:
# Pytorch optimiser
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [41]:
batch_size = 32

for steps in range(10000):
    #sample a batch of data
    xb, yb = get_batch('train')

    #evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

2.5223681926727295


In [42]:
generated_batches = m.generate(idx = torch.zeros((xb.shape[0],1), dtype = torch.long), max_new_tokens = 100)

for generated_batch in generated_batches:
    print(decode(generated_batch.tolist()))





Thaus coph anduponenon mous.
Wee uce Ifo:
Hend; t ce sived hikerad g lelopr knoul, my, t, ot, dot 

Thatr hao su beathe be seaca age s r aronken leteanceneastlllllad ivend wimarsty r geasoupou suke he

APlinghaleat? Y:
Upal n wit thesBulasestofanche





Tothe shiroasXCvelle'dansly, ties g ban at.
Ore

The s wist d pto hirisecens whore omepw these pove achal!
NGrd thindUCE:
Calatiarbe gquso bald ordai


TENT:
Anitindourde'limemen igof o'Sess h s, prs? l qR wh in slisghe t ythise, st tu hoinqRDeareve b

En'lldreran hitarus t ss ayblike ERG bowirar myss;

Anesse d bere, thersunothe, hed mor; y f anoaish

We, withakiond ply, borthu d?

CLAl soore machreit ay y, s; I merinonty. in bes, th we seimad?
rd on

T:
Wigscose d ligealitsofis enemy, y corsspitth.
Tof.
I hTrs urs se mef h,
Wh ton's?
Bocrd co I manc

S owaces blaththerear t;
Whe;
KIXE ghofr ple,
'jug!

O t g hea or tirtaricareagrenc, urQua d buck; n

HAntr.vere onaro oble al, serkean. s lyoslimeatod w.

Tho t off bouse, te V:
Ifay

The mathematical trick in self-attention

In [ ]:
gghhg